In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

# List of DUIDs to plot
duids = ['OSB-AG', 'QPS5', 'PPCCGT', 'TORRB1', 'TORRB4', 'TORRB2', 'TORRB3', 
         'HDWF2', 'LONSDALE', 'HDWF3', 'HDWF1']

# Function to create plot for a single DUID
def plot_duid_bidprices(df, duid):
    """
    Create a plot for a specific DUID showing average bid prices by BIDTYPE
    
    Parameters:
    df (DataFrame): The dataframe containing the bid data
    duid (str): The DUID to plot
    """
    # Filter for the specific DUID
    duid_df = df[df["DUID"] == duid].copy()
    
    if duid_df.empty:
        print(f"No data found for DUID: {duid}")
        return None
    
    # Convert data types for calculations
    duid_df["BIDPRICE"] = pd.to_numeric(duid_df["BIDPRICE"], errors='coerce')
    
    # Ensure SETTLEMENTDATE is datetime
    duid_df["SETTLEMENTDATE"] = pd.to_datetime(duid_df["SETTLEMENTDATE"])
    duid_df.sort_values(by="SETTLEMENTDATE", inplace=True)
    
    # Get unique BIDTYPEs for this DUID
    bid_types = sorted(duid_df["BIDTYPE"].unique())
    
    # Create figure
    plt.figure(figsize=(12, 8))
    
    # Get info about this generator for the title
    if "Participant" in duid_df.columns and "Station Name" in duid_df.columns:
        participant = duid_df["Participant"].iloc[0] if not duid_df["Participant"].iloc[0] is None else "Unknown"
        station = duid_df["Station Name"].iloc[0] if not duid_df["Station Name"].iloc[0] is None else "Unknown"
        title = f"{duid} - {station} ({participant})"
    else:
        title = f"DUID: {duid}"
    
    # Plot each BIDTYPE
    for bid_type in bid_types:
        # Filter for this BIDTYPE
        bid_df = duid_df[duid_df["BIDTYPE"] == bid_type].copy()
        bid_df.set_index("SETTLEMENTDATE", inplace=True)
        
        # Calculate monthly average price
        monthly_avg = bid_df["BIDPRICE"].resample("M").mean()
        
        # Skip if all NaN
        if monthly_avg.isna().all():
            continue
        
        # Plot this BIDTYPE
        plt.plot(
            monthly_avg.index,
            monthly_avg,
            linestyle="-",
            linewidth=2,
            marker='o',
            markersize=4,
            label=bid_type
        )
    
    # Add labels and title
    plt.title(title, fontsize=14)
    plt.ylabel("Average Bid Price ($/MWh)", fontsize=12)
    plt.xlabel("Date", fontsize=12)
    
    # Format x-axis
    plt.xticks(rotation=45)
    
    # Add grid and legend
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(loc='best')
    
    # Adjust layout
    plt.tight_layout()
    
    return plt.gcf()